# Harvesting under a strong Allee effect

<div style="border-left:5px solid #176b87;background:#176b8710;padding:12px"><strong>Read, compute, interpret</strong><br>Five linked work packages study constant catch, proportional effort, collapse boundaries, forward PINNs and inverse inference. All numerical examples are synthetic.</div>

1. WP01 derives scaling, equilibria and stability.
2. WP02 compares explicit harvesting protocols and follows continuous histories with declared restocking.
3. WP03 classifies basins and checks finite extinction by independent quadrature and solvers.
4. WP04 tests forward PINNs against numerical references.
5. WP05 fits identifiable parameter subsets using noisy observations, a classical baseline and held-out windows.

Equations and algorithms precede code; plots are followed by interpretation and limitations. The shared classical APIs use one threshold/branch provider. A small population in B is not finite extinction; A's zero continuation is a declared shutdown policy. Good prediction does not guarantee accurate parameter recovery.

The standalone export includes readable source/config/data/evidence. Restart the kernel and Run All. Recorded final experiments are distinguished from live demonstrations. The inverse reporting fixes from PR #10 commit `00393cd` are integrated. Section 5 explains the legacy training-status label and the separate L-BFGS outcome. The Overleaf report, presentation/rehearsal and human approvals are maintained separately.


In [ ]:
import os
from pathlib import Path

import mm_allee

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src/mm_allee").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout")
os.chdir(ROOT)
print("Package:", mm_allee.__version__)

## 1. Foundations and Model A: steady populations under a fixed catch

**Question.** With the same catch every year, where can the population stay steady, which of those levels are safe, and what is the largest catch before collapse is certain?

**Model A.** A well-mixed population $N(t)$ with growth rate $r$, capacity $K$ and Allee threshold $A$ ($0<A<K$), harvested at a constant rate $H$:
$$\frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)\Big(\frac{N}{A}-1\Big)-H.$$
Below $A$ growth is negative, so a small population shrinks even without fishing. We assume deterministic dynamics and constant parameters, and stop a simulation when $N$ reaches 0, since an extinct stock cannot be fished.

**Scaling.** With $x=N/K$, $\tau=rt$, $a=A/K$ and $q=H/(rK)$,
$$\frac{dx}{d\tau}=f(x)=G(x)-q,\qquad G(x)=x(1-x)\Big(\frac{x}{a}-1\Big),$$
so only $a$ and $q$ matter. We use $a=0.2$, for example $r=0.5$/year, $K=1000$ t and $A=200$ t as in Section 2. The next cell checks the scaling.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mm_allee.models import Parameters, dimensional_rhs, rhs

# dN/dt in tonnes/year should equal rK times dx/dtau
p = Parameters(r=0.5, K=1000.0, A=200.0, H=150.0)  # 1/year, tonnes, tonnes, tonnes/year
d = p.dimensionless()
N = np.array([100.0, 200.0, 500.0, 800.0])
print("a =", d["a"], "  q =", d["q"])
print("dN/dt        :", dimensional_rhs(N, "A", p))
print("rK * dx/dtau :", p.r * p.K * rhs(N / p.K, "A", a=d["a"], harvest=d["q"]))

### 1.1 Steady states and stability

A steady state $x_e$ solves $f(x_e)=0$. It is **stable** if $f>0$ just below it and $f<0$ just above it, so a small disturbance dies out. With the opposite signs it is **unstable**, and with the same sign on both sides it is **semistable**.

**Logistic baseline** ($f=x(1-x)-q$, no Allee effect). For $q<\tfrac14$ the steady states $x_\pm=\tfrac12\big(1\pm\sqrt{1-4q}\big)$ are unstable (lower) and stable (upper). They merge at $q=\tfrac14$, so the largest catch is $H_c=rK/4$. Without harvest, zero is unstable and any small population recovers.

In [ ]:
from mm_allee.equilibria import equilibria_logistic

for q in [0.0, 0.1, 0.25, 0.3]:
    states = [f"x={e.x:.3f} ({e.stability})" for e in equilibria_logistic(q)]
    print(f"q={q}:", ", ".join(states) or "none, the population collapses")

### 1.2 Model A: the growth curve and the maximum catch

$G$ is negative on $(0,a)$ and forms a hill on $(a,1)$. Setting $G'(x)=0$ gives the top of the hill,
$$x^*=\frac{1+a+\sqrt{1-a+a^2}}{3},\qquad q_c=G(x^*).$$
Steady states are where the line at height $q$ meets $G$:

- **$q=0$:** $0$ (stable), $a$ (unstable, the minimum viable population) and $1$ (stable).
- **$0<q<q_c$:** $x_-$ between $a$ and $x^*$ (unstable) and $x_+$ between $x^*$ and 1 (stable). Below $x_-$ the population dies out; above it, it settles at $x_+$.
- **$q=q_c$:** one semistable point at $x^*$, the fold.
- **$q>q_c$:** none, so every population collapses.

For $q>0$, $f(0)=-q<0$, so zero is not a steady state and a declining population reaches zero in finite time. For $a=0.2$, $x^*\approx0.706$ and $q_c\approx0.525$, a maximum catch of about 263 t/year in the example. The next cell checks $x^*$ and $q_c$ against a numerical maximisation of $G$.

In [ ]:
from scipy.optimize import minimize_scalar

from mm_allee.equilibria import equilibria_a, fold_a
from mm_allee.models import growth

a = 0.2
x_star, q_c = fold_a(a)
peak = minimize_scalar(lambda x: -growth(x, a), bounds=(a, 1), method="bounded")
print(f"formula:            x* = {x_star:.4f}, q_c = {q_c:.4f}")
print(f"numerical maximum:  x  = {peak.x:.4f}, G   = {-peak.fun:.4f}")

### 1.3 Finding the steady states numerically

**Algorithm.**
1. Compute $x^*$ and $q_c$.
2. If $q>q_c$ there is no positive steady state; if $q=q_c$ there is one, at $x^*$.
3. If $0<q<q_c$, $f$ changes sign once on $[a,x^*]$ and once on $[x^*,1]$. Find each root with Brent's method, a standard root finder that narrows an interval around a sign change.
4. Label each root from the signs of $f$ on either side, as in 1.1.

In [ ]:
for q in [0.0, 0.3, q_c, 0.6]:
    states = [f"x={e.x:.4f} ({e.stability})" for e in equilibria_a(a, q)]
    print(f"q={q:.4f}:", ", ".join(states) or "none, the population collapses")

### 1.4 Growth curves and the branch diagram

Filled circles are stable, open circles unstable and half-filled semistable. Arrows on the phase lines show whether the population grows or shrinks.

In [ ]:
from mm_allee.equilibria_plots import branch_figure, perturbation_figure, production_figure

fig = production_figure(a=0.2, logistic_q=(0.1, 0.25, 0.35), allee_q=(0.25 * q_c, q_c, 1.2 * q_c))
plt.show()

**Interpretation.** The logistic population has a threshold only when harvested. Model A has one even without harvest ($x=a$), and fishing raises it. Above $q_c$ the catch line misses the hill entirely.

In [ ]:
fig = branch_figure(a=0.2)
plt.show()

**Interpretation.** As the catch rises, the stable branch (solid) and the collapse threshold (dashed) move together and meet at the fold $(q_c,x^*)$. The safety margin between them shrinks like $\sqrt{q_c-q}$.

### 1.5 Checking stability by simulation

At $q=0.3$, start $\delta=0.01$ above and below each steady state and solve the ODE (RK45, stopping at zero).

In [ ]:
fig = perturbation_figure(a=0.2, q=0.3, delta=0.01, tau_end=12.0)
plt.show()

Both starts near $x_+$ return to it. Just above $x_-$ the population recovers, and just below it reaches zero at $\tau\approx2.9$, as the stability labels predict.

<div style="border-left:5px solid #1baf7a;background:#1baf7a14;padding:10px 14px;border-radius:4px"><strong>Main result</strong><br>As a fixed catch rises, the safe steady state and the collapse threshold move together and merge at the maximum catch $q_c$. A disturbance causes collapse only if it is larger than the gap between them, and that gap vanishes at $q_c$, so fishing near the maximum leaves almost no room for a bad year.</div>

**Limitations.** No randomness (which matters most near the threshold), a simple cubic growth law, constant parameters, and an exact catch whenever $N>0$. The numbers are illustrative, not from a real stock.

## 2. Proportional harvesting and recovery

<div style="border-left:5px solid #c87116;background:#c8711614;padding:10px 14px"><strong>Question:</strong> Does reducing the catch restore a collapsed population? Carry state through the whole experiment and distinguish recovery from explicit restocking.</div>

Owner: @AshChan07. These are synthetic model calculations, not observations.

**Question:** does a smaller catch when population declines guarantee recovery?
No: below the Allee threshold, natural growth is already negative.
At 100 tonnes an effort of 0.05/year removes 5 tonnes/year, but at 20 tonnes
it removes 1 tonne/year. We compare this with a constant catch using common units.

## Derivation before computation
With $x=N/K$, $\tau=rt$, $a=A/K$, $e=h/r$,
$$f_B(x)=x[(1-x)(x/a-1)-e].$$
The nonzero roots solve $x^2-(1+a)x+a(1+e)=0$, so
$$x_\pm=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}2,\qquad e_c=\frac{(1-a)^2}{4a}.$$
For $e<e_c$, $f_B=-x(x-x_-)(x-x_+)/a$ has signs $-,+,-$ on the three positive intervals.
Thus the lower positive root is unstable (the basin boundary) and the upper root is stable.
Zero is stable because $f'_B(0)=-1-e<0$. At the fold, the factor is a square:
the field is negative on both sides, so attraction occurs only from above.
Above the threshold only zero remains. At zero effort, recover $0,a,1$.

**Algorithm:** compute the discriminant threshold; return zero; return no positive
roots above it, one double root at it, or two roots below it. Use the product of
roots for the smaller root to avoid cancellation. The `e` field is effort, not
Task 1's yield `q`. Independent high-precision polynomial and trajectory tests
check this implementation.

In [ ]:
import csv
import json
import os
from pathlib import Path

from IPython.display import Image, display

from mm_allee.model_b import critical_effort, equilibria_b
from mm_allee.wp02_experiment import run_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
for effort in [0, 0.4, critical_effort(0.2), 1.0]:
    print(effort, [(r.x, r.stability) for r in equilibria_b(0.2, effort)])

## Fair comparison definitions

1. **Initial catch:** $H=hN_0$. Set $H=\alpha H_c^A$ and solve for $h$; the logistic model also gets $H$.
2. **Threshold fraction:** match $H_A/H_c^A=h/h_c=H_L/H_c^L=\alpha$. Initial catches can differ.

Use $r=0.5$/year, $K=1000$ tonnes, $A=200$ tonnes; $N_0/K=0.15,0.4,0.9$ and $\alpha=0.5,1.2$.
Calculate theoretical outcomes from phase lines separately from the finite observation horizon ($\tau=30$).
Integrate actual B catch over time; stop A/logistic catch at extinction. Divide total catch by the full horizon.
A blank critical population indicates no populated equilibrium. Equal $r,K$ does not match
logistic and Allee production curves, so a universal ecological risk ranking would be unjustified.

### What the yield columns mean

For B, realised catch changes with biomass: $Y_B(t)=hN(t)$, hence
$$C_B=\int_0^T hN(t)\,dt=eK\int_0^{rT}x(\tau)\,d\tau.$$
For A and logistic harvesting, $C=H\min(T,t_{\mathrm{ext}})$ when a finite
extinction event occurs, and $C=HT$ otherwise. The mean catch is $C/T$,
including any time after shutdown; it is not the mean over surviving years only.
The table below uses tonnes for biomass/total catch and tonnes/year for catch rates.
A critical starting biomass separates collapse from persistence; an exact boundary
is an equilibrium, whereas a nearby numerically uncertain start is reported as
`boundary_unresolved`, not forced into either outcome.

## History algorithm

Use staircase controls from 0 to 1.3 times critical and back, with 12 increments each way,
12 units at the peak, and dwell times 1 and 3 on ordinary intervals. Carry every terminal state forward.
After returning to zero, wait 15 units, then compare a 20-unit unassisted hold with a run adding $0.6K$.
For A, record the first zero event and shut down harvest; the zero continuation is a declared policy.
For B, never use the cutoff to reset the state. Report only the first sampled $x\leq10^{-6}$,
not an exact extinction time. Paired boundary rows encode left/right states; only restocking changes population discontinuously.

The following executes the actual bounded CPU experiment and writes a manifest. It also reruns
all four restocking histories with tighter integration settings; errors must be below the
predeclared absolute limits, $2\times10^{-7}$ in population and event time.

In [ ]:
config = json.loads((ROOT / "configs/wp02.json").read_text())
output = ROOT / "results/wp02-notebook"
manifest = run_experiment(config, output, ROOT)
print(manifest["status"], manifest["elapsed_seconds"], "seconds")
display(Image(filename=str(output / "wp02_b_branches.png")))
display(Image(filename=str(output / "wp02_history.png")))

In [ ]:
from IPython.display import Markdown


def show_table(headers, rows):
    text = "| " + " | ".join(headers) + " |\n"
    text += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    text += "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)
    display(Markdown(text))


with (output / "wp02_comparison.csv").open() as f:
    comparison = list(csv.DictReader(f))
display(Markdown("### Fair comparison: start at 900 tonnes, fraction 0.5"))
selected = [
    row
    for row in comparison
    if float(row["initial_population"]) == 900 and float(row["fraction"]) == 0.5
]
show_table(
    [
        "Protocol",
        "Model",
        "Initial catch (t/yr)",
        "Critical start (t)",
        "Total catch (t)",
        "Mean catch (t/yr)",
        "Final biomass (t)",
        "Outcome",
    ],
    [
        [
            row["protocol"],
            row["model"],
            f"{float(row['initial_catch']):.2f}",
            f"{float(row['critical_initial_population']):.2f}"
            if row["critical_initial_population"]
            else "No positive equilibrium",
            f"{float(row['total_catch']):.2f}",
            f"{float(row['mean_catch']):.2f}",
            f"{float(row['final_population']):.2f}",
            row["theoretical_outcome"],
        ]
        for row in selected
    ],
)
display(
    Markdown(
        "These six rows illustrate the two protocols over 60 years. "
        "The complete 36-row comparison, including starts below the Allee threshold "
        "and controls above the fold, is saved in `wp02_comparison.csv`."
    )
)
display(Markdown("### History: collapse, smallness and separate restocking"))
show_table(
    [
        "Model",
        "Dwell (tau)",
        "Finite extinction (yr)",
        "First sampled x <= 1e-6 (yr)",
        "Restock time (yr)",
        "Final unassisted x",
        "Final restocked x",
        "Max refinement error (x)",
    ],
    [
        [
            row["model"],
            row["dwell"],
            f"{row['extinction_tau'] / config['parameters']['r']:.6f}"
            if row["extinction_tau"] is not None
            else "None",
            f"{row['first_sample_below_epsilon_tau'] / config['parameters']['r']:.2f}"
            if row["first_sample_below_epsilon_tau"] is not None
            else "Not observed",
            f"{row['restock_tau'] / config['parameters']['r']:.1f}",
            f"{row['final_unassisted']:.3g}",
            f"{row['final_restocked']:.6f}",
            f"{row['max_refinement_x_error']:.2e}",
        ]
        for row in manifest["summary"]
    ],
)

## Interpretation and limits

At equal initial catch (131.28 tonnes/year), the example from 900 tonnes persists for A/B
but collapses for the logistic model: the catch exceeds that model's 125 tonnes/year maximum.
At half of each threshold, all three persist from 900 tonnes, but catches differ.
This contrast is about calibration, not evidence that Allee growth always improves safety.

For both ramp speeds, unassisted A reaches extinction and B becomes very small.
Reducing harvest to zero fails to restore a population below $a=0.2$.
Adding $0.6K$ at zero harvest crosses this threshold and restores the population toward $K$.
This is history dependence with an intervention, **not a spontaneous closed hysteresis loop**.
A event times in the curated run are about 13.436601 and 34.612578 in $\tau$ units;
B's first sampled smallness times are 20.4 and 42.0. Dynamic delay is not a new equilibrium threshold.
The two schedules also change total exposure, not just ramp speed.

Refinement errors in the curated runs are below $2\times10^{-9}$; tiny B tails below solver
absolute tolerance have no claimed relative accuracy. Test checks include high-precision roots,
independent maximisation, perturbations, matched units, boundary continuity, insufficient and sufficient
restocking, numerical refinement and manifest checksums. See `docs/tasks/WP02_model_b.md` for the evidence map.

The integrated notebook uses current reviewed WP01 code and WP03's shared threshold/basin helpers.
Comparisons retain the validated range $0.05\leq a\leq0.95$ and conservatively exclude ambiguous near-fold inputs.
A near-boundary initial population is explicitly unresolved. Final group slides, human reviews and rehearsal remain pending.
Optional slowing-down experiments are deferred. Codex assisted drafting and implementation;
human understanding has not been certified.

**Presentation check:** why does zero remain stable when effort falls back to zero?
Explain the sign of natural growth below the Allee threshold before quoting a formula.

## 3. Collapse thresholds, finite-time extinction and basins
<div style="border-left:4px solid #16803d;padding:10px"><strong>WP03 analysis implemented.</strong> Synthetic deterministic models; final group integration, human reviews and Devam's detailed learning review remain pending.</div>

For a well-mixed population, assume constant $r>0$, $0<A<K$, nonnegative harvest, no delays/age structure or other species. Put $x=N/K,\ \tau=rt,\ a=A/K,\ q=H/(rK),\ e=h/r$. Then
$$
x'=G(x)-q\quad(A),\qquad x'=G(x)-ex\quad(B),\qquad
G(x)=-x^3/a+(1+a)x^2/a-x.
$$
**Threshold derivation.** $G'=(-3x^2+2(1+a)x-a)/a$. The larger stationary point is
$x_*=(1+a+\sqrt{1-a+a^2})/3$, in $(a,1)$, with $G''(x_*)<0$;
the smaller point lies below a. Thus $q_c=G(x_*)$ and
$N_*=(K+A+\sqrt{K^2-KA+A^2})/3,\ H_c=rKq_c$.
For B, maximise $P(x)=G(x)/x=(1-x)(x/a-1)$:
$P'=(1+a-2x)/a,\ P''=-2/a$, so
$e_c=(1-a)^2/(4a)$, $h_c=re_c$.
The logistic class maximum is $1/4$, giving $H_{c,\mathrm{logistic}}=rK/4$.
H and h have different units; threshold comparisons also depend on production calibration.

**Algorithm:** compute closed-form thresholds; check them against independent bounded numerical maxima of expanded polynomials; classify basins before any extinction integration.

In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

from mm_allee.models import Parameters
from mm_allee.thresholds import dimensional_thresholds

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/wp03.json").exists() and (p / "src/mm_allee").exists()
)
config = json.loads((ROOT / "configs/wp03.json").read_text())
parameters = Parameters(**config["parameters"])
print("Synthetic parameters:", vars(parameters))
print("Dimensional thresholds:", dimensional_thresholds(parameters))

### Basins and critical cases
For $0<q<q_c$, A has two positive roots $x_-<x_+$ and signs $-,+,-$: below the lower root extinction, above it approach to the upper root. Exact roots remain equilibria. For B,
$$
x_\pm=(1+a\pm\sqrt{(1-a)^2-4ae})/2.
$$
At positive roots $f'_B=xP'$, so the lower is unstable and upper stable; zero has derivative $-(1+e)<0$.

| Control | A | B |
|---|---|---|
| Zero | Below a approaches zero asymptotically; above a tends to 1 | Same positive-state basins |
| Below fold | Below lower root reaches zero; above tends to upper root | Below lower root approaches zero; above tends to upper root |
| At fold | Below double root reaches zero; above approaches it | Below double root approaches zero; above approaches it |
| Above fold | Every finite positive start reaches zero | Every positive start approaches zero |

Exact initial equilibria stay there in all rows. At a fold the derivative is zero but signs on both sides are negative: **semistable**, attracting only from above.
The classifier explicitly recognises returned roots and marks non-exact starts within $2\,10^{-12}$ of an unstable/semistable boundary unresolved. It does not merge adjacent harvest controls into the fold or decide survival from a finite horizon.

Numerically, compute the B upper root first and use the quadratic root product `lower = a*(1+e)/upper` to avoid cancellation for small a; the exact fold remains an explicit repeated root. The integrated WP01–WP03 APIs use the same threshold and branch providers and exact equality to the computed critical control. Adjacent controls retain distinct topology; numerical and initial-state uncertainty are reported separately.


### Extinction integral and solver checks
On a reaching-zero A path,
$$
\tau_{\rm ext}=\int_0^{x_0}\frac{du}{q-G(u)},\qquad
t_{\rm ext}=\tau_{\rm ext}/r=\int_0^{N_0}\frac{dN}{H-g(N)}.
$$
For a subcritical start below the barrier the denominator is continuous and positive over the whole path; at/above an intervening equilibrium the extinction integral is inapplicable. For $q>q_c$, $G\le q_c$ gives $\tau_{\rm ext}\le x_0/(q-q_c)$.
For zero-yield A and B, smoothness/uniqueness exclude a positive trajectory reaching zero in finite time.

**Algorithm:** classify -> integrate only reaching-zero paths -> compare quadrature with two RK45 tolerances and independently expanded-polynomial DOP853 -> retain every case/status. No equilibrium pole is crossed. Blank times mean no finite extinction or unresolved boundary, not zero.

RK45 uses a shared-stage 5(4) pair for adaptive error control. We take the fifth-order state, locate the first downward zero event for positive-yield A, and include its endpoint without adding a zero tail. Output times are separate from adaptive internal steps. Quadrature error estimates and local RK tolerances are not rigorous global error bounds. References: [SciPy solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), [quad](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.quad.html).

**Difficult endpoint handling:** near the subcritical barrier, a logarithmic variable removes the vanishing linear factor. At the computed fold an elementary partial-fraction expression evaluates the extinction integral (`fold_closed_form`, no quadrature-error estimate). This avoids missed endpoint contributions for close-below-fold starts. Root rounding is separate from integration error. See `docs/WP03_EXPLAINED.md` for a basics-first explanation and the derivation.

Near a subcritical boundary, $d\tau_{\rm ext}/dx_0 = 1/[q-G(x_0)] \sim 1/[f_A'(x_-)(x_--x_0)]$. Rounding in the initial state or root is amplified as the gap shrinks; small integration error alone does not justify extra digits.


In [ ]:
outputs = {
    "analysis": ROOT / "results/wp03-notebook",
    "reference": ROOT / "results/wp03-reference-notebook",
}
commands = [
    ("analysis", "experiments/wp03/run.py", "configs/wp03.json"),
    ("reference", "experiments/wp03/generate_reference.py", "configs/wp03_reference.json"),
]
for label, script, configuration in commands:
    completed = subprocess.run(
        [
            sys.executable,
            str(ROOT / script),
            "--config",
            str(ROOT / configuration),
            "--output",
            str(outputs[label]),
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())


def read_rows(filename):
    with (outputs["analysis"] / filename).open(newline="") as stream:
        return list(csv.DictReader(stream))


evidence = read_rows("wp03_event_validation.csv")
assert all(row["status"] == "passed" for row in evidence)
print(
    "Largest recorded event difference in dimensionless time:",
    max(float(row["max_abs_error_tau"]) for row in evidence),
)
display(Image(filename=str(outputs["analysis"] / "wp03_basins.png")))
display(Image(filename=str(outputs["analysis"] / "wp03_extinction.png")))

### Interpret the synthetic curves
The default $r=0.7,K=100,A=20$ gives $N_*=70.55050$, $H_c=36.75966$, $h_c=0.56$; logistic gives $17.5$.
The 427 extinction evaluations retain non-finite outcomes rather than turning them into zero. Initial zero is omitted on the logarithmic time axis. Solid basin boundaries mark critical initial populations; the finite colour grid does not replace exact equilibrium classification.

At fixed reaching-zero initial population, increasing catch reduces extinction time. Approaching a basin boundary can make that time very large. A finite simulation horizon without an event does not settle whether extinction happens later. The six event checks include the fold from below and supercritical passage; the observed differences are much smaller than the declared $2\,10^{-7}+5\,10^{-7}\tau_{\rm ext}$ acceptance tolerance.

Basin colour regions meet exactly at control/critical = 1; the fold is drawn as a line, so persistence cells do not extend above it. Every WP03 figure is exported as PNG/EPS/vector PDF; extensionless report references select the PDF under pdflatex.


### Why collapse can look delayed: the bottleneck
With $q=q_c+\delta,\ y=x-x_*$, the exact cubic expansion is
$$
q-G(x)=\delta+\beta y^2+y^3/a,\qquad \beta=\sqrt{1-a+a^2}/a.
$$
For a fixed start above $x_*$, the trajectory crosses a slow region. Integrating the leading denominator gives
$$
\int\frac{dy}{\delta+\beta y^2}
=\frac{\arctan(y\sqrt{\beta/\delta})}{\sqrt{\beta\delta}},
\quad \tau_{\rm ext}\sim\pi/\sqrt{\beta\delta}.
$$
The full near-fold integral is evaluated with $y=\sqrt{\delta/\beta}\tan\theta$ to resolve its narrow peak.
This is passage after the equilibria disappear, distinct from relaxation to a stable equilibrium. A start already below the bottleneck need not have this divergence. At the fold, approach from above is algebraic, $y\simeq y_0/(1+\beta y_0\tau)$.

In [ ]:
rows = read_rows("wp03_bottleneck.csv")
for row in rows:
    print("delta/qc =", row["fraction_above_fold"], "| tau*sqrt(delta) =", row["scaled_time"])
print("Leading scaled-time limit:", rows[0]["asymptotic_scaled_time"])
display(Image(filename=str(outputs["analysis"] / "wp03_bottleneck.png")))

### Optional results: sensitivity and sustainable yield
Holding other dimensional r,K,A fixed, $g_N(N_*)=0$ eliminates the derivative of $N_*$ from the maximum:
$$
H_{c,r}=H_c/r,\quad H_{c,K}=rN_*^2(N_*/A-1)/K^2,\quad
H_{c,A}=-rN_*^2(1-N_*/K)/A^2.
$$
Also
$$
h_{c,r}=h_c/r,\quad h_{c,K}=\tfrac r4(1/A-A/K^2),\quad
h_{c,A}=\tfrac r4(-K/A^2+1/K).
$$
Central differences check these derivatives; elasticities are $(p/C)C_p$. An a sweep at fixed r,K is a different experiment.

A's stable-branch yield supremum $H_c$ lies at the semistable cliff, with no control margin.
For B, $Y(h)=hN_+(h)=g(N_+(h))$. The production maximiser is on the upper branch, so
$h_{\rm MSY}=g(N_*)/N_*<h_c,\ Y_{\rm MSY}=H_c$.
Independent bounded numerical maximisation checks the optimum. The example effort margin at MSY is about 6.96%; the initial state must still exceed the unstable root. Separate illustrative A/B policies are not a universal risk ranking.

In [ ]:
summary = json.loads((outputs["analysis"] / "wp03_summary.json").read_text())
print(json.dumps(summary, indent=2))
display(Image(filename=str(outputs["analysis"] / "wp03_sensitivity_yield.png")))
display(Image(filename=str(outputs["reference"] / "wp03_rk45_reference.png")))

### Handoff, reproduction and limitations
The early handoff retains six clean A/B CSVs (606 points). WP05 owns noise, frozen splits, inverse PINN/ODE fitting and held-out validation; WP04 supplies the shared neural engine. Keep known conditions/fixed external scales separate from evaluation truth, and never fit against held-out clean values or hidden unknown parameters.

Exact commands and test/evidence mapping: **docs/WP03_ANALYSIS.md**. The experiment regenerates tables and PNG/EPS; manifests record versions, units, tolerances and source/file hashes. All data are synthetic; no empirical species-level conclusion follows.

The report, notebook and EPS files required by the course must ultimately be integrated with all five tasks. **scripts/check_submission_ready.py** aggregates missing final artifacts, unfinished sections and metadata; a successful WP03 run does not make the final group submission ready. Reviews and non-author reproduction remain required.

AI assistance: OpenAI ChatGPT/Codex for this implementation, mathematical writing, tests and documentation, recorded in docs/AI_USAGE.md. Devam's detailed understanding review remains pending as requested.